# Databricks 개요

## 목차
1. [Databricks란 무엇인가](#1-databricks란-무엇인가)
2. [왜 Databricks를 사용하는가](#2-왜-databricks를-사용하는가)
3. [데이터 아키텍처의 진화](#3-데이터-아키텍처의-진화)
4. [Lakehouse Architecture](#4-lakehouse-architecture)
5. [Databricks Workspace 구조](#5-databricks-workspace-구조)

---
## 1. Databricks란 무엇인가

**Databricks**는 Apache Spark를 기반으로 만들어진 **클라우드 통합 데이터 플랫폼**이다.  
2013년 Apache Spark를 만든 UC Berkeley 연구팀이 창업했으며, 현재 AWS / Azure / GCP 위에서 동작한다.

### 한 줄 정의
> **데이터 엔지니어링 + 데이터 사이언스 + ML + AI를 하나의 플랫폼에서 처리하는 Lakehouse 플랫폼**

### Databricks가 해결하는 문제

| 기존 문제 | Databricks 해결 방식 |
|-----------|---------------------|
| 데이터 저장소와 분석 도구가 분리되어 있다 | 저장과 처리를 하나의 플랫폼에서 처리 |
| 대용량 데이터 처리를 위한 인프라 구축이 어렵다 | 클라우드 기반 완전 관리형 Spark 제공 |
| DE / DS / ML 팀이 서로 다른 도구를 사용한다 | Notebook 기반 협업 환경 통합 |
| 데이터 거버넌스가 분산되어 관리가 어렵다 | Unity Catalog로 중앙 통합 거버넌스 |
| 데이터 품질과 ACID 보장이 어렵다 | Delta Lake로 안정적인 트랜잭션 처리 |

---
## 2. 왜 Databricks를 사용하는가

### 2-1. 기존 접근 방식의 한계

데이터 규모가 커지면서 단일 서버 기반 RDB로는 처리가 불가능한 상황이 발생한다.

```
[ 문제 상황 ]

데이터 1TB → 단일 MySQL 서버 → 쿼리 수십 분 ~ 수 시간
데이터 100TB → 불가능
```

### 2-2. Databricks의 접근 방식

```
[ Databricks 방식 ]

데이터 1TB   → 10개 노드 분산 처리 → 수 분
데이터 100TB → 100개 노드로 확장   → 동일하게 수 분
```

수평 확장(Scale-out)을 통해 데이터 크기에 관계없이 일정한 처리 시간을 유지한다.

---
## 3. 데이터 아키텍처의 진화

Databricks가 왜 등장했는지 이해하려면 데이터 아키텍처의 흐름을 알아야 한다.

### 3-1. Data Warehouse

```
[ 구조 ]
원천 시스템 → ETL → Data Warehouse (정형 테이블) → BI 도구
```

**특징**
- 정형 데이터(테이블)만 저장 가능
- SQL 기반 빠른 분석 쿼리
- ACID 트랜잭션 보장

**한계**
- 비정형 데이터(이미지, 로그, JSON) 저장 불가
- 스토리지 비용이 매우 높음
- ML/AI 워크로드에 부적합

### 3-2. Data Lake

```
[ 구조 ]
원천 시스템 → Raw 데이터 그대로 → S3 / ADLS / GCS (파일 저장)
```

**특징**
- 정형 / 반정형 / 비정형 데이터 모두 저장 가능
- 저렴한 오브젝트 스토리지 활용 (S3 등)
- ML/AI 워크로드에 유연하게 대응

**한계**
- ACID 트랜잭션 없음 → 데이터 불일치 발생
- 데이터 품질 보장 어려움
- 거버넌스 / 권한 관리 복잡
- 관리가 안 되면 **Data Swamp(데이터 늪)** 로 전락

### 3-3. Data Lakehouse ← Databricks의 포지션

```
[ 구조 ]
원천 시스템 → Delta Lake (S3 위에 트랜잭션 레이어 추가) → 분석 + ML 통합
```

**Data Warehouse의 장점 + Data Lake의 장점을 결합**

| 특성 | Data Warehouse | Data Lake | **Lakehouse** |
|------|---------------|-----------|---------------|
| 저장 비용 | 높음 | 낮음 | **낮음** |
| ACID 트랜잭션 | O | X | **O** |
| 비정형 데이터 | X | O | **O** |
| ML/AI 지원 | X | O | **O** |
| 데이터 품질 | 높음 | 낮음 | **높음** |
| 거버넌스 | 강함 | 약함 | **강함** |

---
## 4. Lakehouse Architecture

### 4-1. 전체 구조

```
┌─────────────────────────────────────────────────────────┐
│                      Databricks                          │
│                                                          │
│   Notebooks │ Jobs │ Delta Live Table │ ML │ SQL         │
│                                                          │
│              Apache Spark (실행 엔진)                     │
│                                                          │
│                Delta Lake (스토리지 레이어)                │
│                                                          │
│          Cloud Storage  (S3 / ADLS / GCS)                │
└─────────────────────────────────────────────────────────┘
                    ↕ 메타데이터 관리
                 Unity Catalog
```

### 4-2. 핵심 기술 스택

**Apache Spark**
- Databricks의 실행 엔진
- 여러 노드에 데이터를 분산하여 병렬 처리
- Driver(작업 조율) + Executor(실제 연산) 구조
- Lazy Evaluation: Action 호출 전까지 실행 계획만 수립

**Delta Lake**
- Parquet 파일 + 트랜잭션 로그(_delta_log)의 조합
- ACID 보장, Time Travel, Schema Enforcement 제공
- Databricks의 기본 저장 포맷

**Unity Catalog**
- 카탈로그 > 스키마 > 테이블 3단계 네임스페이스
- 중앙 집중식 권한 관리, Data Lineage, 감사 로그

### 4-3. Medallion Architecture

데이터를 품질 단계별로 분리하여 관리하는 설계 패턴

```
원천 데이터
    ↓
[ Bronze ] ─── 원본 데이터 그대로 저장 (Raw)
               변환 없이 적재, 재처리 기준점
    ↓
[ Silver ] ─── 정제 및 표준화 (Cleansed)
               타입 변환, 중복 제거, 결측값 처리
    ↓
[ Gold ]   ─── 집계 및 비즈니스 로직 적용 (Aggregated)
               분석/ML 목적에 맞게 가공된 최종 데이터
```

| 레이어 | 목적 | 특징 |
|--------|------|------|
| Bronze | 원본 보존 | 변환 없음, 재처리 가능 |
| Silver | 데이터 품질 확보 | 정제, 표준화, 검증 |
| Gold | 분석/ML 사용 | 집계, 비즈니스 로직 적용 |

---
## 5. Databricks Workspace 구조

### 5-1. Workspace

Databricks의 작업 공간. 노트북, 파이프라인, 모델, 파일 등 모든 자산을 관리한다.

- 팀 단위로 Workspace를 분리하거나 공유 가능
- 폴더 구조로 Notebook / 파일 / 실험 관리
- Git 연동 (Repo) 지원

### 5-2. Cluster

Spark를 실행하는 컴퓨팅 자원. Driver 1대 + Worker N대로 구성된다.

```
[ Cluster 구조 ]

  Driver Node
  ├── 작업 계획 수립 (DAG 생성)
  ├── 태스크를 Executor에 분배
  └── 결과 수집

  Worker Node 1 (Executor)
  Worker Node 2 (Executor)   ← 데이터 병렬 처리
  Worker Node N (Executor)
```

| 유형 | 용도 |
|------|------|
| All-purpose | 개발 / 탐색용, Notebook에서 대화형 실행 |
| Job Cluster | 자동화 Job 전용, 실행 후 자동 종료 |
| SQL Warehouse | SQL 분석 전용 컴퓨팅 |
| Serverless | 인프라 없이 즉시 사용 |

### 5-3. Notebook

코드와 문서를 함께 작성하는 인터랙티브 실행 환경

- Python / SQL / Scala / R 지원
- 같은 Notebook에서 언어를 혼합 사용 가능 (`%sql`, `%python`, `%scala`)
- `display()` 함수로 DataFrame을 시각화
- Cluster에 연결하여 실행

```python
# Python 셀
df = spark.read.csv("/path/to/file.csv", header=True)
display(df)
```

```sql
-- SQL 셀 (%sql 매직 커맨드)
%sql
SELECT * FROM my_table LIMIT 10
```

### 5-4. Job

Notebook이나 파이프라인을 자동으로 실행하는 스케줄러

- Cron 표현식 또는 이벤트 트리거로 스케줄 설정
- 실행 시마다 Job Cluster를 새로 생성 → 실행 후 자동 종료 (비용 효율)
- 실행 이력 / 로그 / 알림 관리
- 멀티 태스크 Job으로 파이프라인 오케스트레이션 가능

```
[ Job 흐름 예시 ]

매일 오전 6시
  → Job Cluster 생성
  → Task 1: Bronze 적재 Notebook 실행
  → Task 2: Silver 변환 Notebook 실행  (Task 1 완료 후)
  → Task 3: Gold 집계 Notebook 실행   (Task 2 완료 후)
  → Job Cluster 자동 종료
```

### 5-5. Repo

Git 저장소를 Databricks Workspace에 직접 연동

- GitHub / GitLab / Bitbucket 지원
- Notebook을 코드로 버전 관리
- 브랜치 생성 / 커밋 / PR을 Workspace에서 직접 수행
- CI/CD 파이프라인과 연동 가능

### 5-6. SQL Warehouse

SQL 분석 전용 컴퓨팅 리소스

- BI 도구 (Tableau, Power BI 등) 연동에 최적화
- Serverless 또는 Classic 모드 선택 가능
- 쿼리 결과 캐싱으로 반복 쿼리 성능 향상
- Databricks SQL Editor에서 직접 쿼리 실행

```
[ Cluster vs SQL Warehouse 선택 기준 ]

Python / 복잡한 변환 / ML → All-purpose Cluster
SQL 분석 / BI 연동         → SQL Warehouse
자동화 파이프라인           → Job Cluster
```

---
## 확인 실습

아래 코드를 실행하여 현재 접속한 Databricks 환경을 확인한다.

In [ ]:
# Spark 버전 및 현재 환경 확인
print(f"Spark Version  : {spark.version}")
print(f"App Name       : {spark.conf.get('spark.app.name')}")
print(f"현재 카탈로그  : {spark.catalog.currentCatalog()}")
print(f"현재 스키마    : {spark.catalog.currentDatabase()}")
print(f"현재 사용자    : {spark.sql('SELECT current_user()').collect()[0][0]}")

In [ ]:
# 사용 가능한 카탈로그 목록 확인
display(spark.sql("SHOW CATALOGS"))

In [ ]:
# 카탈로그 변경 후 스키마 목록 확인
spark.sql("USE CATALOG <카탈로그명>")
display(spark.sql("SHOW SCHEMAS"))